# Module 2: Benchmark Foundation Models at Scale

⏱ **Estimated time:** ~45–60 minutes

In this notebook, you'll score foundation models on the dataset from Module 1, using a judge model to grade each answer. The scores you save here are the baseline that the following modules compare against.

## What You'll Do

1. **Part 1:** Score 3 models with Amazon Bedrock Evaluations, using its built-in metrics
2. **Part 2:** Score 14 models with Inspect AI, using a legal rubric
3. Compare how the ranking changes between the two rubrics
4. Break the scores down by area of law
5. Save the results to S3

---
Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.  
SPDX-License-Identifier: MIT-0

## Step 0: Install the Notebook Dependencies

Run this first to install the required packages for the notebook.

In [ ]:
# Install the notebook's dependencies (requirements-notebooks.txt sits beside the notebook)
get_ipython().run_line_magic("pip", "install -q -r requirements-notebooks.txt")
print("✅ Dependencies installed.")

## Configuration

Sets up the variables and AWS resources used in this notebook.

In [ ]:
# Settings, AWS clients, and bucket names and the evaluation role from the stack outputs
import boto3
import json
import os
import time
import logging
import pandas as pd
from datetime import datetime, timezone

# Let Inspect AI run inside Jupyter
import nest_asyncio
nest_asyncio.apply()
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

# Settings: the AWS region, and which dataset to evaluate against
REGION = os.environ.get("AWS_REGION", "us-east-1")
DOMAIN = "legal"
JURISDICTION = "au"
DATASET_KEY = f"{DOMAIN}/eval-{DOMAIN}-{JURISDICTION}.jsonl"

def stack_outputs():
    """Every CloudFormation output in the region, by key."""
    # The CloudFormation stack records the names of the resources it created
    cf = boto3.client("cloudformation", region_name=REGION)
    return {o["OutputKey"]: o["OutputValue"]
            for page in cf.get_paginator("describe_stacks").paginate()
            for s in page["Stacks"] for o in s.get("Outputs", [])}

# Look up the dataset and results buckets, and the IAM role that
# Bedrock Evaluations uses to read and write them
outputs = stack_outputs()
DATASET_BUCKET = outputs["DatasetBucketName"]
RESULTS_BUCKET = outputs["ResultsBucketName"]
ROLE_ARN = outputs["EvaluationRoleArn"]

bedrock = boto3.client("bedrock", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

print(f"Dataset: s3://{DATASET_BUCKET}/{DATASET_KEY}")

The next cell sets which models are scored and which model acts as the judge.

| Setting | What it is | Default |
|---|---|---|
| `MANAGED_MODELS` | Models scored by Bedrock Evaluations | 3 models |
| `CANDIDATE_MODELS` | Models scored by Inspect AI | All 14 models |
| `EVALUATOR_MODEL`, `INSPECT_GRADER_MODEL` | The judge model, set once for each tool | Claude Sonnet 4.6 for both |

- **Bedrock Evaluations scores three models** because each job takes several
  minutes. Inspect AI scores all 14, and later modules compare against its results.
- **Both tools use the same judge**, so any difference in their results comes from
  the rubric, not the judge.

If you're short on time, you can comment out models in `CANDIDATE_MODELS`. 

> ❗️ Keep Nova Micro `us.amazon.nova-micro-v1:0`, this is the baseline model used by all modules to compare scores across different techniques.

In [ ]:
# Models to score, the judge, and the Bedrock Evaluations metrics
# How many questions to score per model, and how many requests to send at once
NUM_RECORDS = 50
MAX_CONNECTIONS = 8

# A judge model is an LLM that reads each answer and grades it against the reference answer
# The judge. It is not a candidate, so it never grades its own answers.
EVALUATOR_MODEL = "us.anthropic.claude-sonnet-4-6"
INSPECT_GRADER_MODEL = f"bedrock/{EVALUATOR_MODEL}"

# Part 1: models scored by the managed Bedrock Evaluations service
MANAGED_MODELS = [
    "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "us.amazon.nova-pro-v1:0",
    "us.amazon.nova-micro-v1:0",
]

# Part 2: models scored with Inspect AI, an open-source evaluation framework
CANDIDATE_MODELS = [
    "us.amazon.nova-pro-v1:0",
    "us.amazon.nova-lite-v1:0",
    "us.amazon.nova-micro-v1:0",
    "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "us.anthropic.claude-haiku-4-5-20251001-v1:0",
    "us.meta.llama4-maverick-17b-instruct-v1:0",
    "us.meta.llama3-3-70b-instruct-v1:0",
    "mistral.mistral-large-3-675b-instruct",
    "mistral.ministral-3-14b-instruct",
    "openai.gpt-oss-120b-1:0",
    "openai.gpt-oss-20b-1:0",
    "nvidia.nemotron-super-3-120b",
    "nvidia.nemotron-nano-3-30b",
    "nvidia.nemotron-nano-12b-v2",
]

# Built-in metrics the Bedrock Evaluations judge scores each answer on (written by AWS, not editable)
BEDROCK_METRICS = ["Builtin.Correctness", "Builtin.Completeness", "Builtin.Helpfulness"]

print(f"Judge:  {EVALUATOR_MODEL}")
print(f"Part 1: {len(MANAGED_MODELS)} models via Bedrock Evaluations")
print(f"Part 2: {len(CANDIDATE_MODELS)} models via Inspect AI, {NUM_RECORDS} items each")

---
## Part 1: Bedrock Evaluations (Managed, LLM-as-a-Judge)

Bedrock runs each candidate model over the dataset and has the judge score every answer
for Correctness, Completeness and Helpfulness. Results appear in the console with
a written explanation per answer.

**What this rubric assesses:** Correctness, Completeness and Helpfulness are
built-in metrics written by AWS. They're general-purpose, so they don't
check whether an answer is specific to Australian law.

> ❗️ Bedrock Evaluations jobs can take 5-10 minutes.

### Submit Evaluation Jobs

One evaluation job is created per model. The jobs run in parallel on AWS, so you can carry on with
Part 2 while they run.

In [ ]:
# Start one Bedrock Evaluations job per model
def submit_managed_eval(model_id):
    # Build a short, unique job name from the model name and the current time
    name = model_id.replace("us.", "").split(".")[1].split(":")[0][:20]
    job_name = f"sovereign-{name}-{datetime.now().strftime('%m%d-%H%M%S')}"
    # Point the job at the evaluation dataset in S3
    dataset = {"name": "sovereign-legal-benchmark",
               "datasetLocation": {"s3Uri": f"s3://{DATASET_BUCKET}/{DATASET_KEY}"}}
    try:
        # Ask the candidate model to answer every question, have the judge score the answers,
        # and write the results to the results bucket
        response = bedrock.create_evaluation_job(
            jobName=job_name,
            roleArn=ROLE_ARN,
            applicationType="ModelEvaluation",
            evaluationConfig={"automated": {
                "datasetMetricConfigs": [{"taskType": "General", "dataset": dataset,
                                          "metricNames": BEDROCK_METRICS}],
                "evaluatorModelConfig": {"bedrockEvaluatorModels": [{"modelIdentifier": EVALUATOR_MODEL}]}}},
            inferenceConfig={"models": [{"bedrockModel": {"modelIdentifier": model_id}}]},
            outputDataConfig={"s3Uri": f"s3://{RESULTS_BUCKET}/results/managed/{job_name}/"},
        )
        print(f"  OK   {model_id}")
        return {"model": model_id, "job_name": job_name, "arn": response["jobArn"]}
    # Report a failure (for example, no access to the model) and carry on with the others
    except Exception as e:
        print(f"  FAIL {model_id}: {e}")

# Submit a job for each model and keep only the ones that started
managed_jobs = [j for j in map(submit_managed_eval, MANAGED_MODELS) if j]
print(f"\n{len(managed_jobs)}/{len(MANAGED_MODELS)} jobs submitted. They take ~5-10 minutes.")

### Check Job Progress

Run the cell below as needed to check the progress of the Bedrock Evaluation jobs. You can also watch the jobs
in the console under **Amazon Bedrock → Evaluations**.

In [ ]:
# Show each job's status
if not managed_jobs:
    print("No jobs were submitted.")
# Status is usually InProgress for the first few minutes, then Completed or Failed
for job in managed_jobs:
    info = bedrock.get_evaluation_job(jobIdentifier=job["arn"])
    print(f"  {info['status']:<11} {job['model']}")
    if info["status"] == "Failed":
        print(f"              {'; '.join(info.get('failureMessages', []))}")

---
## Part 2: Inspect AI (Open Source, Legal Rubric)

[Inspect AI](https://inspect.aisi.org.uk) is an open-source evaluation framework
from the UK AI Security Institute. Here it runs every model in `CANDIDATE_MODELS`
and has the judge grade each answer against a **legal rubric** you write.

### Define the Evaluation Task

An Inspect `Task` combines a dataset, a solver (how the candidate answers) and a
scorer (how answers are graded). The scorer, `model_graded_qa`, uses the rubric
below.

**Where this rubric comes from:** you define it yourself in
`LEGAL_INSTRUCTIONS` in the cell below: plain text instructions telling the judge
what counts as a correct answer. This sample gives you a starting version.
Unlike the built-in metrics in Part 1, you decide every criterion, so you can
explain why each answer passed or failed.

> ❗️ In production, ensure this criteria is written with a domain expert.

In [ ]:
# Define the Inspect AI task: dataset, solver and rubric-based scorer
from inspect_ai import Task, task, eval as inspect_eval
from inspect_ai.dataset import FieldSpec, json_dataset
from inspect_ai.model import get_model, GenerateConfig
from inspect_ai.solver import generate
from inspect_ai.scorer import model_graded_qa

# The judge's rubric - you define this yourself. Have a go editing it below!
LEGAL_INSTRUCTIONS = """Grade the submitted answer against the reference answer for a question about Australian law.

Mark the answer CORRECT only if ALL of the following hold:
- It reaches the correct legal conclusion (e.g. liable/not liable, valid/void).
- It cites the correct legislation or section where the reference answer does.
- It is specific to Australian law, not generic or another jurisdiction's law.

Mark it INCORRECT if the legal conclusion is wrong, the cited provision is wrong
or fabricated, or the answer is generic rather than Australian-specific.

First explain your reasoning, then end with your grade in exactly this format:
"GRADE: C" if correct or "GRADE: I" if incorrect."""

# Temperature 0 makes the judge's grading as consistent as possible between runs
grader = get_model(INSPECT_GRADER_MODEL, config=GenerateConfig(temperature=0.0))

# A task bundles the dataset, a solver (how the model produces its answer) and a scorer
# (how the answer is graded)
@task
def au_legal_benchmark():
    return Task(
        # Map each JSONL record's fields: the question is the input, the reference answer is the target
        dataset=json_dataset(
            f"s3://{DATASET_BUCKET}/{DATASET_KEY}",
            FieldSpec(input="prompt", target="referenceResponse", metadata=["category"]),
            limit=NUM_RECORDS,
        ),
        # generate() just sends the question to the model and takes its reply as the answer
        solver=generate(),
        # The judge grades each answer against the reference using the rubric above
        scorer=model_graded_qa(instructions=LEGAL_INSTRUCTIONS, model=grader),
    )

print("✅ Ready to proceed!")

### Run the Evaluation Across Every Model

Inspect calls every model through the same Bedrock API, so one list of model IDs
is all it needs. Each answer is graded by the LLM-as-a-judge.

In [ ]:
# Run the task against every candidate model
print(f"Scoring {len(CANDIDATE_MODELS)} models on {NUM_RECORDS} items each...")
# Run the same task once per model. Logs are saved to ./inspect-logs for later inspection
# Temperature 0 asks each model for its most likely answer, so results are easier to compare
logs = inspect_eval(
    au_legal_benchmark(),
    model=[f"bedrock/{m}" for m in CANDIDATE_MODELS],
    log_dir="./inspect-logs",
    max_tokens=1024,
    max_connections=MAX_CONNECTIONS,
    temperature=0.0,
)
print(f"\nDone: {len(logs)} models scored.")

### Collect Per-Model and Per-Category Scores

Each answer is marked correct (1) or incorrect (0). This flattens the results into
one row per answer.

In [ ]:
# Collect each model's score per answer (correct = 1, incorrect = 0)
inspect_results = []
# Each log holds one model's run
for log in logs:
    model_id = log.eval.model.replace("bedrock/", "")
    # Record a placeholder for models whose run failed, so they're not silently dropped
    if log.status != "success" or not log.samples:
        inspect_results.append({"model": model_id, "category": None, "score": None})
        continue
    for sample in log.samples:
        # The judge's grade is "C" for correct or "I" for incorrect
        score_obj = next(iter(sample.scores.values())) if sample.scores else None
        value = 1.0 if (score_obj and str(score_obj.value).upper().startswith("C")) else 0.0
        inspect_results.append({
            "model": model_id,
            "category": (sample.metadata or {}).get("category", "unknown"),
            "score": value,
        })

print(f"Collected {len(inspect_results)} graded samples across {len(logs)} models.")

---
## Part 3: Collect Bedrock Evaluation Results

The cell below waits for the Bedrock Evaluation jobs to complete (checking every 30 seconds), then downloads each
answer's scores and the judge's explanation from S3.

In [ ]:
# Wait for the Bedrock Evaluations jobs, then download their results
# Check every 30 seconds until each job finishes, giving up after 30 minutes
pending, start = list(managed_jobs), time.time()
while pending and time.time() - start < 1800:
    for job in pending:
        info = bedrock.get_evaluation_job(jobIdentifier=job["arn"])
        job["status"] = info["status"]
        if job["status"] not in ("InProgress", "Created"):
            reason = "; ".join(info.get("failureMessages", []))
            print(f"  [{int(time.time() - start)}s] {job['status']}: {job['model']} {reason}")
    # Keep checking only the jobs that haven't finished yet
    pending = [j for j in pending if j["status"] in ("InProgress", "Created")]
    if pending:
        time.sleep(30)

# Read each completed job's results from S3
managed_results = []
for job in managed_jobs:
    if job.get("status") != "Completed":
        continue
    # Each job writes its results to its own folder in the results bucket
    listing = s3.list_objects_v2(Bucket=RESULTS_BUCKET, Prefix=f"results/managed/{job['job_name']}/")
    for obj in listing.get("Contents", []):
        if obj["Key"].endswith("_output.jsonl"):
            body = s3.get_object(Bucket=RESULTS_BUCKET, Key=obj["Key"])["Body"].read().decode()
            # Each line of the output file is one question with its scores for each metric
            for line in body.strip().splitlines():
                data = json.loads(line)
                scores = {s["metricName"]: s["result"] for s in data["automatedEvaluationResult"]["scores"]}
                managed_results.append({"model": job["model"],
                                        "category": data["inputRecord"].get("category", "unknown"), **scores})

print(f"Collected {len(managed_results)} Bedrock Evaluations results.")

---
## Part 4: Combined Scorecard

View the results of both evaluation/rubric mechanisms together.

- **Inspect AI:** legal correctness (0–1) per model, with a per-category breakdown
- **Bedrock Evaluations:** Correctness, Completeness and Helpfulness for the three
  managed models

Compare where the two rankings agree and where they don't.

In [ ]:
# Print both scorecards
# Leave out models whose run failed
df_inspect = pd.DataFrame([r for r in inspect_results if r["score"] is not None])
# Average score per model, from best to worst
print("=== Inspect AI: legal correctness (0-1) ===\n")
print(df_inspect.groupby("model")["score"].mean().sort_values(ascending=False).to_string())
# Average score per model in each area of law
print("\n=== Inspect AI: by category ===\n")
print(df_inspect.pivot_table(index="model", columns="category", values="score", aggfunc="mean").to_string())

# Average of each Bedrock Evaluations metric per model, sorted by correctness
if managed_results:
    df_managed = pd.DataFrame(managed_results)
    print("\n=== Bedrock Evaluations ===\n")
    print(df_managed.groupby("model")[BEDROCK_METRICS].mean()
          .sort_values(BEDROCK_METRICS[0], ascending=False).to_string())

---
## Part 5: Save Results

Saves both scorecards to S3 with a timestamp. It also saves Nova Micro's score
as the baseline that the following modules compare against.

In [ ]:
# Save both scorecards to S3
# A timestamp in the file names keeps each run's results separate
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
def put_json(key, data):
    s3.put_object(Bucket=RESULTS_BUCKET, Key=key, Body=json.dumps(data).encode(),
                  ContentType="application/json")

put_json(f"results/inspect/scorecard-{timestamp}.json", inspect_results)
if managed_results:
    put_json(f"results/managed/scorecard-{timestamp}.json", managed_results)

# Save Nova Micro's score as the baseline for Modules 4, 5 and 6
BASELINE_MODEL = "us.amazon.nova-micro-v1:0"
# Number Nova Micro's graded answers 1, 2, 3, ... so later modules can compare them
# question by question
per_item = [{"id": i, "category": r["category"], "score": r["score"]}
            for i, r in enumerate((r for r in inspect_results
                                   if r["model"] == BASELINE_MODEL and r["score"] is not None), 1)]
if not per_item:
    raise SystemExit(f"{BASELINE_MODEL} wasn't scored, so there's no baseline. Add it back to CANDIDATE_MODELS.")
# The baseline is the share of questions Nova Micro got right
baseline = sum(p["score"] for p in per_item) / len(per_item)
# Save the baseline with details of how it was scored, for the comparison tables in later modules
put_json(f"results/cells/none-base-{timestamp}.json", {
    "cell": "none", "model_label": "base", "model_id": f"bedrock/{BASELINE_MODEL}",
    "score": baseline, "n": len(per_item), "judge": EVALUATOR_MODEL,
    "eval_key": DATASET_KEY, "notebook": "Module-2-Evaluation",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "config": {"retrieval": None}, "per_item": per_item,
})
print(f"✓ Scorecards saved. Nova Micro baseline: {baseline:.3f} (n={len(per_item)})")

---
## Key Takeaways

- **Bedrock Evaluations** is the managed path: built-in metrics, results in the
  console, a written explanation per answer.
- **Inspect AI** is the open-source path: a rubric you write, results as data, and you
  can add custom scorers.
- **A judge, not word overlap:** "liable" and "not liable" share almost every word.
  A rubric-driven judge catches the difference.
- Changing the rubric can change the ranking, so publish your rubric with your scores.

## Success Criteria

Before moving on, verify that you have:

- [ ] 3 Bedrock Evaluations jobs submitted and collected
- [ ] 14 models scored by Inspect AI with a per-category breakdown
- [ ] A combined scorecard showing both paths side by side
- [ ] Both scorecards saved to `results/inspect/` and `results/managed/` in S3
- [ ] Identified which categories scored worst: you will watch these later

## Next Steps

Open **Module-4-Fine-Tune-Results.ipynb** to see how a fine-tuned Nova Micro
scored. Module 3, how that model was trained, is a set of CloudShell steps rather
than a notebook (`docs/MODULE-3-FINE-TUNE.md`), because it runs in us-east-1
and training takes about 45 minutes.